In [ ]:
import numpy as np
import pandas as pd
import random
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import imageio.v2 as imageio
import os

# --- 1. Configuration Parameters ---

# Network Structure
NUM_FACTORIES = 1
NUM_RDCS = 2
RETAILERS_PER_RDC_DISTRIBUTION = [2, 3] # RDC_0 serves 2 retailers, RDC_1 serves 3 retailers
NUM_RETAILERS = sum(RETAILERS_PER_RDC_DISTRIBUTION)

# Demand Generation Parameters
DEMAND_MEAN_RETAILER = 4
DEMAND_MEAN_RDC = 80
DEMAND_MEAN_FACTORY = 150

# Lead Times (in days)
LEAD_TIME_FACTORY_TO_RDC = 7
LEAD_TIME_RDC_TO_RETAILER = 3

# Initial Inventory Levels
INITIAL_INVENTORY_FACTORY = 500
INITIAL_INVENTORY_RDC = 100
INITIAL_INVENTORY_RETAILER = 12

# Reorder Point (s) and Order Up-to Level (S/Q)
REORDER_POINT_RETAILER = 5
ORDER_UP_TO_LEVEL_RETAILER = 20

REORDER_POINT_RDC = 50
ORDER_UP_TO_LEVEL_RDC = 150

REORDER_POINT_FACTORY = 200
ORDER_UP_TO_LEVEL_FACTORY = 800

# Simulation duration
SIMULATION_DAYS = 30

# Cost Structure
HOLDING_COST = {
    'factory': 0.1,
    'rdc': 0.5,
    'retailer': 2.0
}
ORDERING_COST = {
    'factory': 1000,
    'rdc': 200,
    'retailer': 10
}
STOCKOUT_COST = {
    'factory': 100,
    'rdc': 50,
    'retailer': 10
}


# --- 2. Network Setup ---

def setup_network():
    """Sets up the NetworkX graph for visualization and defines node positions."""
    G = nx.DiGraph()
    pos = {}

    # Add Factory Nodes
    factory_id = f'factory_0'
    G.add_node(factory_id, type='Factory')
    pos[factory_id] = (0, 0)

    # Add RDC Nodes and edges from Factory to RDCs
    rdc_y_positions = {
        f'rdc_0': 3.5,
        f'rdc_1': -3.5
    }
    for rdc_idx in range(NUM_RDCS):
        rdc_id = f'rdc_{rdc_idx}'
        G.add_node(rdc_id, type='RDC')
        pos[rdc_id] = (1, rdc_y_positions[rdc_id])
        G.add_edge(f'factory_0', rdc_id)

    # Add Retailer Nodes and edges from RDCs to Retailers
    retailer_y_offsets_per_rdc = {
        f'rdc_0': [5, 2],
        f'rdc_1': [-2, -5, -8]
    }

    for rdc_idx in range(NUM_RDCS):
        num_retailers_for_this_rdc = RETAILERS_PER_RDC_DISTRIBUTION[rdc_idx]
        for ret_idx in range(num_retailers_for_this_rdc):
            retailer_id = f'retailer_{rdc_idx}_{ret_idx}'
            G.add_node(retailer_id, type='Retailer')
            pos[retailer_id] = (2, retailer_y_offsets_per_rdc[f'rdc_{rdc_idx}'][ret_idx])
            G.add_edge(f'rdc_{rdc_idx}', retailer_id)

    # Define node colors based on type
    node_colors = []
    for node in G.nodes():
        if G.nodes[node]['type'] == 'Factory':
            node_colors.append('skyblue')
        elif G.nodes[node]['type'] == 'RDC':
            node_colors.append('lightgreen')
        else:
            node_colors.append('lightcoral')

    return G, pos, node_colors


# --- 3. Simulation Function ---

def simulate_day(
    current_day,
    inventories_on_hand,
    inventories_in_transit,
    backlogs,
    total_costs
):
    """
    Simulates one day of operations for the entire supply chain network.
    Updates inventory, in-transit, backlog, and cumulative costs.
    """
    daily_holding_cost = 0
    daily_ordering_cost = 0
    daily_stockout_cost = 0
    orders_placed_this_day = {} # To track ordering costs for the current day

    # --- Factory Operations ---
    for f_idx in range(NUM_FACTORIES):
        factory_id = f'factory_{f_idx}'

        # 1. Process Incoming Orders (conceptual: RDC orders trigger factory shipments)
        # This is handled when RDCs place orders and factory stock is deducted.

        # 2. Ordering/Production Decision (Factory)
        current_factory_stock = inventories_on_hand.get(factory_id, 0)
        factory_in_transit_qty = sum(qty for (origin, dest), shipments in inventories_in_transit.items()
                                     if origin == factory_id and dest.startswith('rdc_')
                                     for arrival_day, qty in shipments if arrival_day > current_day)

        if current_factory_stock + factory_in_transit_qty <= REORDER_POINT_FACTORY:
            order_qty = ORDER_UP_TO_LEVEL_FACTORY - (current_factory_stock + factory_in_transit_qty)
            if order_qty > 0:
                orders_placed_this_day[factory_id] = orders_placed_this_day.get(factory_id, 0) + order_qty
                daily_ordering_cost += ORDERING_COST['factory']
                inventories_on_hand[factory_id] += order_qty # Instant production for now, can be refined

        # Holding Cost for Factory
        daily_holding_cost += current_factory_stock * HOLDING_COST['factory']

    # --- RDC Operations ---
    for rdc_idx in range(NUM_RDCS):
        rdc_id = f'rdc_{rdc_idx}'

        # 1. Process Incoming Shipments (Factory to RDC)
        if (f'factory_0', rdc_id) in inventories_in_transit:
            for i, (arrival_day, quantity) in enumerate(inventories_in_transit[(f'factory_0', rdc_id)]):
                if arrival_day == current_day:
                    inventories_on_hand[rdc_id] += quantity
                    inventories_in_transit[(f'factory_0', rdc_id)][i] = None
            inventories_in_transit[(f'factory_0', rdc_id)] = [s for s in inventories_in_transit[(f'factory_0', rdc_id)] if s is not None]

        # 2. Ordering Decision (RDC to Factory)
        current_rdc_stock = inventories_on_hand.get(rdc_id, 0)
        rdc_in_transit_qty = sum(qty for arrival_day, qty in inventories_in_transit.get((f'factory_0', rdc_id), []) if arrival_day > current_day)

        if current_rdc_stock + rdc_in_transit_qty <= REORDER_POINT_RDC:
            order_quantity = ORDER_UP_TO_LEVEL_RDC - (current_rdc_stock + rdc_in_transit_qty)
            if order_quantity > 0:
                arrival_day = current_day + LEAD_TIME_FACTORY_TO_RDC
                if inventories_on_hand.get(f'factory_0', 0) >= order_quantity:
                    inventories_in_transit.setdefault((f'factory_0', rdc_id), []).append((arrival_day, order_quantity))
                    inventories_on_hand[f'factory_0'] -= order_quantity # Deduct from factory stock immediately
                    orders_placed_this_day[rdc_id] = orders_placed_this_day.get(rdc_id, 0) + order_quantity
                    daily_ordering_cost += ORDERING_COST['rdc']
                # else: factory stockout, RDC order not placed/backlogged (simplified)

        # Holding Cost for RDC
        daily_holding_cost += current_rdc_stock * HOLDING_COST['rdc']


    # --- Retailer Operations ---
    for rdc_idx in range(NUM_RDCS):
        num_retailers_for_this_rdc = RETAILERS_PER_RDC_DISTRIBUTION[rdc_idx]
        for ret_idx in range(num_retailers_for_this_rdc):
            retailer_id = f'retailer_{rdc_idx}_{ret_idx}'
            serving_rdc_id = f'rdc_{rdc_idx}'

            # 1. Demand Generation
            daily_demand_retailer = np.random.poisson(DEMAND_MEAN_RETAILER)

            # 2. Process Incoming Shipments (RDC to Retailer)
            if (serving_rdc_id, retailer_id) in inventories_in_transit:
                for i, (arrival_day, quantity) in enumerate(inventories_in_transit[(serving_rdc_id, retailer_id)]):
                    if arrival_day == current_day:
                        inventories_on_hand[retailer_id] += quantity
                        inventories_in_transit[(serving_rdc_id, retailer_id)][i] = None
                inventories_in_transit[(serving_rdc_id, retailer_id)] = [s for s in inventories_in_transit[(serving_rdc_id, retailer_id)] if s is not None]

            # 3. Fulfill Demand and Update Inventory
            current_retailer_stock = inventories_on_hand.get(retailer_id, 0)
            current_backlog = backlogs.get(retailer_id, 0)

            # First, fulfill any existing backlog
            if current_backlog > 0:
                if current_retailer_stock >= current_backlog:
                    inventories_on_hand[retailer_id] -= current_backlog
                    backlogs[retailer_id] = 0
                    current_retailer_stock = inventories_on_hand[retailer_id]
                else:
                    backlogs[retailer_id] -= current_retailer_stock
                    current_retailer_stock = 0
                    inventories_on_hand[retailer_id] = 0

            # Now, fulfill today's demand
            if current_retailer_stock >= daily_demand_retailer:
                inventories_on_hand[retailer_id] -= daily_demand_retailer
            else:
                fulfilled_demand = current_retailer_stock
                stockout = daily_demand_retailer - current_retailer_stock
                backlogs[retailer_id] = backlogs.get(retailer_id, 0) + stockout
                inventories_on_hand[retailer_id] = 0
                daily_stockout_cost += stockout * STOCKOUT_COST['retailer']

            # 4. Ordering Decision (Retailer to RDC)
            current_retailer_stock = inventories_on_hand.get(retailer_id, 0)
            retailer_in_transit_qty = sum(qty for arrival_day, qty in inventories_in_transit.get((serving_rdc_id, retailer_id), []) if arrival_day > current_day)

            if current_retailer_stock + retailer_in_transit_qty <= REORDER_POINT_RETAILER:
                order_quantity = ORDER_UP_TO_LEVEL_RETAILER - (current_retailer_stock + retailer_in_transit_qty)
                if order_quantity > 0:
                    arrival_day = current_day + LEAD_TIME_RDC_TO_RETAILER
                    if inventories_on_hand.get(serving_rdc_id, 0) >= order_quantity:
                        inventories_in_transit.setdefault((serving_rdc_id, retailer_id), []).append((arrival_day, order_quantity))
                        inventories_on_hand[serving_rdc_id] -= order_quantity # Deduct from RDC stock immediately
                        orders_placed_this_day[retailer_id] = orders_placed_this_day.get(retailer_id, 0) + order_quantity
                        daily_ordering_cost += ORDERING_COST['retailer']
                    # else: RDC stockout, retailer order not placed/backlogged (simplified)

            # Holding Cost for Retailer
            daily_holding_cost += current_retailer_stock * HOLDING_COST['retailer']

    # Aggregate daily costs
    total_costs['holding'] += daily_holding_cost
    total_costs['ordering'] += daily_ordering_cost
    total_costs['stockout'] += daily_stockout_cost

    return inventories_on_hand, inventories_in_transit, backlogs, total_costs


# --- 4. Visualization Function ---

def plot_daily_state(
    day,
    inventories_on_hand,
    inventories_in_transit,
    backlogs,
    G, # NetworkX graph
    pos, # Node positions for NetworkX
    node_colors, # Node colors for NetworkX
    total_costs_till_day,
    inventory_history_all, # Full inventory history for sawtooth plots
    simulation_days_total # Total simulation days for x-axis scaling
):
    """
    Generates a plot of the supply chain network for a single day, showing inventory levels,
    in-transit shipments, and embedded sawtooth plots for each node's inventory history.
    """
    fig = plt.figure(figsize=(25, 15))
    ax_network = fig.add_subplot(111)

    # Network Graph
    nx.draw_networkx_nodes(G, pos, node_color=node_colors, node_size=3000, alpha=0.9, ax=ax_network)
    nx.draw_networkx_edges(G, pos, edge_color='gray', arrows=True, arrowsize=20, ax=ax_network)

    node_labels = {}
    for node_id in G.nodes():
        inv = inventories_on_hand.get(node_id, 0)
        label = f"{node_id}\nInv: {inv}"
        if node_id in backlogs and backlogs[node_id] > 0:
            label += f"\nBacklog: {backlogs[node_id]}"
        node_labels[node_id] = label
    nx.draw_networkx_labels(G, pos, labels=node_labels, font_size=12, font_weight='bold', ax=ax_network)

    # Display in-transit shipments
    for (origin, dest), shipments in inventories_in_transit.items():
        # Filter for shipments that are actually in transit (not yet arrived)
        current_day_shipments_qty = sum(qty for arrival_day, qty in shipments if arrival_day > day)
        if current_day_shipments_qty > 0:
            x1, y1 = pos[origin]
            x2, y2 = pos[dest]
            mid_x, mid_y = (x1 + x2) / 2, (y1 + y2) / 2
            offset_x = (y2 - y1) * 0.05
            offset_y = (x1 - x2) * 0.05
            ax_network.text(mid_x + offset_x, mid_y + offset_y,
                            f'IT: {current_day_shipments_qty}',
                            color='blue', fontsize=9, bbox=dict(facecolor='white', alpha=0.7, edgecolor='none'))

    ax_network.set_title(f"Network State - Day {day}", size=14)
    ax_network.axis('off')

    # Embedded Sawtooth Plots
    ordered_node_ids = []
    for f_idx in range(NUM_FACTORIES):
        ordered_node_ids.append(f'factory_{f_idx}')
    for rdc_idx in range(NUM_RDCS):
        ordered_node_ids.append(f'rdc_{rdc_idx}')
        num_retailers_for_this_rdc = RETAILERS_PER_RDC_DISTRIBUTION[rdc_idx]
        for ret_idx in range(num_retailers_for_this_rdc):
            ordered_node_ids.append(f'retailer_{rdc_idx}_{ret_idx}')

    for node_id in ordered_node_ids:
        node_x, node_y = pos[node_id]

        # Default inset position to the right of the node
        bbox_to_anchor = [node_x + 0.1, node_y - 0.05, 0.2, 0.1]
        loc = 'center left'
        plot_title = f'{node_id} Inv'

        if 'factory' in node_id:
            bbox_to_anchor = [node_x - 0.05, node_y + 0.15, 0.2, 0.1]
            loc = 'lower center'
        elif 'rdc' in node_id:
            if node_y > 0:
                 bbox_to_anchor = [node_x + 0.1, node_y + 0.05, 0.2, 0.1]
                 loc = 'lower left'
            else:
                bbox_to_anchor = [node_x + 0.1, node_y - 0.15, 0.2, 0.1]
                loc = 'upper left'
        elif 'retailer' in node_id:
            bbox_to_anchor = [node_x - 0.1, node_y - 0.05, 0.2, 0.1]
            loc = 'center right'

        ax_inset = inset_axes(ax_network, width=2.5, height=1.5, loc=loc,
                              bbox_to_anchor=bbox_to_anchor,
                              bbox_transform=ax_network.transData,
                              borderpad=0.5)

        if day > 0 and len(inventory_history_all[node_id]) >= day:
            ax_inset.plot(range(1, day + 1), inventory_history_all[node_id][:day], marker='.', linestyle='-', markersize=2)
        else:
            ax_inset.plot([], [])

        ax_inset.set_title(plot_title, fontsize=7)
        ax_inset.set_xlabel('Day', fontsize=6)
        ax_inset.set_ylabel('Level', fontsize=6)
        ax_inset.tick_params(axis='both', which='major', labelsize=5)
        ax_inset.grid(True, linestyle='--', alpha=0.7)
        ax_inset.set_xlim(1, simulation_days_total) # Use total simulation days for consistent x-axis
        if inventory_history_all[node_id]:
            max_inv_for_node = max(inventory_history_all[node_id])
            ax_inset.set_ylim(0, max_inv_for_node * 1.1)
        else:
            ax_inset.set_ylim(0, 100)

    fig.suptitle(f"Supply Chain Simulation - Day {day} | Total Cost: ${total_costs_till_day['holding'] + total_costs_till_day['ordering'] + total_costs_till_day['stockout']:.2f}",
                 fontsize=16, y=1.02)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])
    plt.savefig(f'simulation_frames/day_{day:03d}.png')
    plt.close(fig)


# --- 5. Main Simulation Execution ---

def run_full_simulation():
    """Initializes state, runs the daily simulation loop, and generates visualizations."""

    # Initialize Network
    G, pos, node_colors = setup_network()

    # Initialize State Variables
    inventories_on_hand = {
        f'factory_0': INITIAL_INVENTORY_FACTORY
    }
    for rdc_idx in range(NUM_RDCS):
        inventories_on_hand[f'rdc_{rdc_idx}'] = INITIAL_INVENTORY_RDC
        num_retailers_for_this_rdc = RETAILERS_PER_RDC_DISTRIBUTION[rdc_idx]
        for ret_idx in range(num_retailers_for_this_rdc):
            inventories_on_hand[f'retailer_{rdc_idx}_{ret_idx}'] = INITIAL_INVENTORY_RETAILER

    inventories_in_transit = {}
    backlogs = {}
    for rdc_idx in range(NUM_RDCS):
        num_retailers_for_this_rdc = RETAILERS_PER_RDC_DISTRIBUTION[rdc_idx]
        for ret_idx in range(num_retailers_for_this_rdc):
            backlogs[f'retailer_{rdc_idx}_{ret_idx}'] = 0

    total_costs = {
        'holding': 0.0,
        'ordering': 0.0,
        'stockout': 0.0
    }

    # List to store daily states for animation
    daily_states = []

    # Initialize inventory history for sawtooth plots
    inventory_history = {}
    # Collect all node IDs for inventory tracking
    all_node_ids = []
    for f_idx in range(NUM_FACTORIES):
        all_node_ids.append(f'factory_{f_idx}')
    for rdc_idx in range(NUM_RDCS):
        all_node_ids.append(f'rdc_{rdc_idx}')
        num_retailers_for_this_rdc = RETAILERS_PER_RDC_DISTRIBUTION[rdc_idx]
        for ret_idx in range(num_retailers_for_this_rdc):
            all_node_ids.append(f'retailer_{rdc_idx}_{ret_idx}')

    for node_id in all_node_ids:
        inventory_history[node_id] = []

    print(f"\nStarting simulation for {SIMULATION_DAYS} days...")

    # Simulation Loop
    for day in range(1, SIMULATION_DAYS + 1):
        inventories_on_hand, inventories_in_transit, backlogs, total_costs = simulate_day(
            current_day=day,
            inventories_on_hand=inventories_on_hand,
            inventories_in_transit=inventories_in_transit,
            backlogs=backlogs,
            total_costs=total_costs
        )

        # Capture the state at the end of each day
        daily_states.append({
            'day': day,
            'inventories_on_hand': inventories_on_hand.copy(),
            'inventories_in_transit': {k: v[:] for k, v in inventories_in_transit.items()},
            'backlogs': backlogs.copy(),
            'total_costs_till_day': total_costs.copy()
        })

        # Populate inventory history for the current day's state
        for node_id in all_node_ids:
            inventory_history[node_id].append(inventories_on_hand.get(node_id, 0))

    print(f"Simulation finished after {SIMULATION_DAYS} days.")

    # Display Simulation Results
    print("\n--- Simulation Results ---")
    print(f"Final Total Holding Cost: ${total_costs['holding']:.2f}")
    print(f"Final Total Ordering Cost: ${total_costs['ordering']:.2f}")
    print(f"Final Total Stockout Cost: ${total_costs['stockout']:.2f}")
    print(f"Overall Total Cost: ${sum(total_costs.values()):.2f}")

    print("\n--- Final Inventory Levels ---")
    for node_id, inv in inventories_on_hand.items():
        print(f"  {node_id}: {inv} units")

    print("\n--- Final Backlog Levels ---")
    has_backlog = False
    for node_id, backlog_qty in backlogs.items():
        if backlog_qty > 0:
            print(f"  {node_id}: {backlog_qty} units")
            has_backlog = True
    if not has_backlog:
        print("  No backlogs remaining.")

    print("\n--- Final In-Transit Shipments ---")
    has_in_transit = False
    for (origin, dest), shipments in inventories_in_transit.items():
        if shipments:
            print(f"  {origin} to {dest}: {shipments}")
            has_in_transit = True
    if not has_in_transit:
        print("  No shipments in transit.")

    # --- 6. Generate GIF from Frames ---

    # Create a directory to save daily frames
    if not os.path.exists('simulation_frames'):
        os.makedirs('simulation_frames')
    else:
        # Clear previous frames if directory exists
        for file in os.listdir('simulation_frames'):
            os.remove(os.path.join('simulation_frames', file))

    print("\nGenerating daily frames with integrated sawtooth plots...")
    for state in daily_states:
        day = state['day']
        inventories_on_hand_day = state['inventories_on_hand']
        inventories_in_transit_day = state['inventories_in_transit']
        backlogs_day = state['backlogs']
        total_costs_till_day = state['total_costs_till_day']

        plot_daily_state(day, inventories_on_hand_day, inventories_in_transit_day, backlogs_day,
                         G, pos, node_colors, total_costs_till_day, inventory_history, SIMULATION_DAYS)

    print(f"Generated {len(daily_states)} frames.")

    print("Creating GIF...")
    images = []
    for day in range(1, SIMULATION_DAYS + 1):
        filename = f'simulation_frames/day_{day:03d}.png'
        images.append(imageio.imread(filename))

    gif_filename = 'supply_chain_simulation.gif'
    imageio.mimsave(gif_filename, images, fps=2)
    print(f"GIF '{gif_filename}' created in your Colab environment.")

    from IPython.display import Image, display
    display(Image(filename=gif_filename))
    print("Displayed the simulation GIF in the notebook.")

    # --- 7. Plot Final Inventory Sawtooth Graphs ---

    print("\nGenerating final inventory sawtooth plots...")
    num_nodes = len(all_node_ids)
    num_cols = 3
    num_rows = (num_nodes + num_cols - 1) // num_cols

    plt.figure(figsize=(num_cols * 6, num_rows * 4))
    plt.suptitle("Inventory Levels Over Time (Sawtooth Plots)", fontsize=16)

    for i, node_id in enumerate(all_node_ids):
        plt.subplot(num_rows, num_cols, i + 1)
        plt.plot(range(1, SIMULATION_DAYS + 1), inventory_history[node_id], marker='.', linestyle='-')
        plt.title(f'{node_id} Inventory')
        plt.xlabel('Day')
        plt.ylabel('Inventory Level')
        plt.grid(True)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    plt.show()
    print("Inventory sawtooth plots generated.")

# Execute the full simulation
run_full_simulation()
